In [3]:
import torch
import torch.nn as nn
import spacy
import numpy as np

# 加载德语、英语分词模型
spacy_de = spacy.load("de_core_news_sm")
spacy_en = spacy.load("en_core_web_sm")

print("spaCy 模型加载成功！")

spaCy 模型加载成功！


In [4]:
# 德语、英语分词函数
def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)]

def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]

# 德英翻译数据集
data = [
    ("ich liebe dich", "i love you"),
    ("es regnet", "it is raining"),
    ("du bist klug", "you are smart"),
    ("wir lernen", "we are learning"),
    ("das buch ist gut", "the book is good")
]

print("数据准备完成！")

数据准备完成！


In [6]:
# 构建单词→编号 字典
SRC = []
TRG = []
for de, en in data:
    SRC.extend(tokenize_de(de))
    TRG.extend(tokenize_en(en))

src_vocab = list(set(SRC))
trg_vocab = list(set(TRG))

src2idx = {w:i+2 for i,w in enumerate(src_vocab)}
trg2idx = {w:i+2 for i,w in enumerate(trg_vocab)}

# 加入特殊符号
src2idx["<PAD>"] = 0
src2idx["<SOS>"] = 1
trg2idx["<PAD>"] = 0
trg2idx["<SOS>"] = 1

idx2trg = {i:w for w,i in trg2idx.items()}

print("词典构建完成！")
print("德语词汇量：", len(src_vocab))
print("英语词汇量：", len(trg_vocab))

词典构建完成！
德语词汇量： 14
英语词汇量： 13


In [8]:
# 句子 → 编号序列
def encode_sent(sent, vocab, tokenizer):
    tokens = tokenizer(sent)
    return [1] + [vocab[t] for t in tokens]

max_len = 6

# 构建张量
src_tensor = torch.zeros(len(data), max_len).long()
trg_tensor = torch.zeros(len(data), max_len).long()

for i, (de, en) in enumerate(data):
    de_ids = encode_sent(de, src2idx, tokenize_de)
    en_ids = encode_sent(en, trg2idx, tokenize_en)
    src_tensor[i,:len(de_ids)] = torch.LongTensor(de_ids)
    trg_tensor[i,:len(en_ids)] = torch.LongTensor(en_ids)

print("句子编码完成！")
print("输入张量形状：", src_tensor.shape)

句子编码完成！
输入张量形状： torch.Size([5, 6])


In [9]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        
    def forward(self, x):
        x = self.embedding(x)
        outputs, (h, c) = self.lstm(x)
        return h, c

print("编码器定义完成！")

编码器定义完成！


In [11]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)
        
    def forward(self, x, h, c):
        x = self.embedding(x.unsqueeze(1))
        output, (h, c) = self.lstm(x, (h, c))
        output = self.fc(output.squeeze(1))
        return output, h, c

print("解码器定义完成！")

解码器定义完成！


In [13]:
INPUT_DIM = len(src2idx)
OUTPUT_DIM = len(trg2idx)
EMB_DIM = 32
HID_DIM = 64

# 初始化模型
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM)

# 优化器 + 损失
optimizer = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()))
criterion = nn.CrossEntropyLoss()

print("模型初始化完成！")

模型初始化完成！


In [15]:
print("开始训练...")

for epoch in range(200):
    h, c = enc(src_tensor)
    loss = 0
    input_tok = trg_tensor[:, 0]
    
    for t in range(max_len - 1):
        output, h, c = dec(input_tok, h, c)
        loss += criterion(output, trg_tensor[:, t+1])
        input_tok = trg_tensor[:, t+1]
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch: {epoch:3d} | Loss: {loss.item():.3f}")

print("训练完成！")

开始训练...
Epoch:   0 | Loss: 0.144
Epoch:  20 | Loss: 0.119
Epoch:  40 | Loss: 0.100
Epoch:  60 | Loss: 0.086
Epoch:  80 | Loss: 0.075
Epoch: 100 | Loss: 0.066
Epoch: 120 | Loss: 0.059
Epoch: 140 | Loss: 0.053
Epoch: 160 | Loss: 0.048
Epoch: 180 | Loss: 0.043
训练完成！


In [16]:
def translate_sentence(sentence):
    with torch.no_grad():
        tokens = tokenize_de(sentence)
        ids = [1] + [src2idx[t] for t in tokens]
        ids = torch.LongTensor(ids).unsqueeze(0)
        h, c = enc(ids)
        input_tok = torch.LongTensor([1])
        res = []
        
        for _ in range(max_len):
            output, h, c = dec(input_tok, h, c)
            pred = output.argmax(1).item()
            if pred == 0:
                break
            res.append(idx2trg[pred])
            input_tok = torch.LongTensor([pred])
            
        return ' '.join(res[1:])

# 测试翻译
print("==== 翻译结果 ===")
print("德语：ich liebe dich")
print("英语：", translate_sentence("ich liebe dich"))

==== 翻译结果 ===
德语：ich liebe dich
英语： love you
